# Classificação de Sons Ambientais com Deep Learning
## Fine-tuning do Audio Spectrogram Transformer no ESC-50

**Disciplina:** Aprendizado Profundo  
**Professor:** [Tiago Maritan Ugulino de Araujo](http://www.ufpb.br/docente/tiagomaritan)  
**Discentes:**

- Matheus Bruno da Silva Oliveira
- Micael Oliveira de Lima Toscano
- Sergio Caua dos Santos

---

### Resumo

Este trabalho aplica *transfer learning* à classificação de sons ambientais. Um **Audio Spectrogram Transformer (AST)** pré-treinado no AudioSet é adaptado para reconhecer as 50 classes do **ESC-50**, conjunto composto por 2.000 gravações de cinco segundos. O experimento utiliza os folds 1–4 para treinamento e o fold 5 para avaliação. O modelo alcançou **92,75% de acurácia** e **92,64% de F1 macro** no conjunto avaliado.

| Métrica | Resultado |
|---|---:|
| Acurácia | **92,75%** |
| F1 macro | **92,64%** |
| Loss | **0,2183** |
| Tempo de treinamento | **64,8 minutos** |


## Sumário

1. Fundamentação teórica
2. Ambiente e reprodutibilidade
3. Dataset e protocolo experimental
4. Pré-processamento
5. Configuração do AST
6. Fine-tuning
7. Resultados e análise
8. Inferência
9. Limitações e conclusões


## 1. Fundamentação teórica

### 1.1 Forma de onda e taxa de amostragem

Um áudio digital pode ser representado por uma sequência de amplitudes. A taxa de amostragem informa quantas medições são registradas por segundo. O ESC-50 é distribuído em 44,1 kHz, enquanto o AST utilizado espera entradas em 16 kHz; por isso, todos os exemplos são reamostrados antes do processamento.

### 1.2 Espectrograma Log-Mel

A Transformada de Fourier de Curto Termo permite observar como o conteúdo de frequência varia ao longo do tempo. As frequências são projetadas na escala Mel e a energia é convertida para escala logarítmica, produzindo uma representação bidimensional adequada à percepção auditiva.

### 1.3 Audio Spectrogram Transformer

O AST divide o espectrograma em pequenos *patches* e os processa com camadas de autoatenção. Neste projeto, os pesos aprendidos no AudioSet são reutilizados, enquanto a cabeça de classificação original é substituída por uma camada com 50 saídas, uma para cada classe do ESC-50.


## 2. Ambiente e reprodutibilidade

A semente aleatória é fixada em 42. O código seleciona automaticamente CUDA, MPS ou CPU. O treinamento está desativado por padrão para evitar sua repetição acidental; quando o checkpoint local está disponível, ele é reutilizado.


In [ ]:
from pathlib import Path
import random

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import torch

from datasets import Audio, DatasetDict, load_dataset
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from transformers import (
    ASTForAudioClassification,
    AutoConfig,
    AutoFeatureExtractor,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
EXECUTAR_TREINAMENTO = False

diretorio_atual = Path.cwd()
PROJECT_ROOT = diretorio_atual.parent if diretorio_atual.name == "notebooks" else diretorio_atual
MODEL_DIR = PROJECT_ROOT / "models" / "ast_esc50"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ast_esc50"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Dispositivo selecionado: {DEVICE}")
print(f"Treinamento habilitado: {EXECUTAR_TREINAMENTO}")


## 3. Dataset e protocolo experimental

O ESC-50 contém 2.000 clipes, distribuídos uniformemente entre 50 classes e cinco folds oficiais. Neste experimento, os folds 1–4 são destinados ao treinamento e o fold 5 à avaliação. Assim, cada classe possui 32 exemplos no treino e 8 na avaliação.


In [ ]:
dataset_completo = load_dataset("ashraq/esc50")["train"]

print(dataset_completo)
print("Colunas:", dataset_completo.column_names)
print("Quantidade de classes:", len(set(dataset_completo["target"])))


### 3.1 Reamostragem e divisão dos folds

A coluna de áudio é reamostrada para 16 kHz, mantendo os cinco segundos de duração. A função de filtragem recebe somente a coluna `fold`, evitando decodificação desnecessária durante a criação das divisões.


In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
dataset_completo = dataset_completo.cast_column(
    "audio",
    Audio(sampling_rate=feature_extractor.sampling_rate),
)

dataset = DatasetDict({
    "train": dataset_completo.filter(
        lambda fold: fold != 5,
        input_columns=["fold"],
        desc="Separando folds 1–4 para treino",
    ),
    "evaluation": dataset_completo.filter(
        lambda fold: fold == 5,
        input_columns=["fold"],
        desc="Separando fold 5 para avaliação",
    ),
})

audio_verificacao = dataset["train"][0]["audio"]
assert audio_verificacao["sampling_rate"] == 16_000
assert len(audio_verificacao["array"]) == 80_000
print(dataset)


### 3.2 Classes e balanceamento

O mapeamento entre identificadores e categorias é salvo na configuração do modelo. As asserções abaixo verificam o total de classes e o balanceamento das divisões.


In [ ]:
pares_ordenados = sorted(
    set(zip(dataset["train"]["target"], dataset["train"]["category"])),
    key=lambda par: par[0],
)
labels = [categoria for _, categoria in pares_ordenados]
num_labels = len(labels)
label2id = {label: indice for indice, label in enumerate(labels)}
id2label = {indice: label for indice, label in enumerate(labels)}

contagens_treino = np.bincount(dataset["train"]["target"], minlength=num_labels)
contagens_avaliacao = np.bincount(dataset["evaluation"]["target"], minlength=num_labels)

assert num_labels == 50
assert np.all(contagens_treino == 32)
assert np.all(contagens_avaliacao == 8)
print(f"Classes: {num_labels} | treino: {len(dataset['train'])} | avaliação: {len(dataset['evaluation'])}")


## 4. Pré-processamento

O `AutoFeatureExtractor` converte os vetores de áudio em bancos de filtros Log-Mel. O AST recebe matrizes com 1.024 frames temporais e 128 bandas de frequência.


In [ ]:
def preprocess_function(exemplos):
    audio_arrays = [
        np.asarray(audio["array"], dtype=np.float32)
        for audio in exemplos["audio"]
    ]
    return feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
    )

colunas_para_remover = [
    coluna for coluna in dataset["train"].column_names if coluna != "target"
]
encoded_dataset = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=colunas_para_remover,
    desc="Extraindo espectrogramas Log-Mel",
)
encoded_dataset = encoded_dataset.rename_column("target", "labels")
encoded_dataset.set_format("torch")
print("Formato de uma entrada:", tuple(encoded_dataset["train"][0]["input_values"].shape))


## 5. Configuração do AST

Quando o checkpoint final está disponível e o treinamento permanece desativado, ele é carregado diretamente. Caso contrário, o modelo público pré-treinado é inicializado com uma nova cabeça de 50 classes.


In [ ]:
if MODEL_DIR.exists() and not EXECUTAR_TREINAMENTO:
    feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR)
    model = ASTForAudioClassification.from_pretrained(MODEL_DIR)
    origem_modelo = "checkpoint local treinado"
else:
    config = AutoConfig.from_pretrained(
        MODEL_ID,
        num_labels=num_labels,
        label2id=label2id,
        id2label=id2label,
    )
    model = ASTForAudioClassification.from_pretrained(
        MODEL_ID,
        config=config,
        ignore_mismatched_sizes=True,
    )
    origem_modelo = "AST pré-treinado no AudioSet"

parametros_totais = sum(parametro.numel() for parametro in model.parameters())
print("Origem:", origem_modelo)
print(f"Parâmetros: {parametros_totais:,}")


## 6. Fine-tuning

O treinamento utiliza três épocas, taxa de aprendizado de $3\times10^{-5}$, lotes de quatro exemplos e acumulação de gradientes por quatro passos, resultando em lote efetivo de 16. O melhor checkpoint é selecionado pela acurácia de avaliação.

> **Atenção:** o treinamento original já foi concluído. Para reproduzi-lo, altere `EXECUTAR_TREINAMENTO` para `True` na seção 2.


In [ ]:
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(eval_pred.label_ids, predictions),
        "f1_macro": f1_score(
            eval_pred.label_ids,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_steps=30,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    seed=SEED,
    data_seed=SEED,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["evaluation"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

if EXECUTAR_TREINAMENTO:
    resultado_treino = trainer.train()
    trainer.save_model(str(MODEL_DIR))
    feature_extractor.save_pretrained(str(MODEL_DIR))
    trainer.save_state()
    print(f"Treinamento concluído em {resultado_treino.metrics['train_runtime'] / 60:.1f} minutos.")
else:
    print("Treinamento não executado; o checkpoint existente será utilizado.")


## 7. Resultados e análise

Os valores abaixo foram registrados na execução original com semente 42. A melhora na terceira época indica convergência sem aumento da loss de avaliação.

| Época | Acurácia | F1 macro | Loss |
|---:|---:|---:|---:|
| 1 | 91,50% | 91,24% | 0,3896 |
| 2 | 91,75% | 90,98% | 0,2988 |
| 3 | **92,75%** | **92,64%** | **0,2183** |


In [ ]:
epocas = [1, 2, 3]
acuracias = [0.9150, 0.9175, 0.9275]
f1_macros = [0.9124, 0.9098, 0.9264]
perdas = [0.3896, 0.2988, 0.2183]

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
eixos[0].plot(epocas, acuracias, marker="o", label="Acurácia")
eixos[0].plot(epocas, f1_macros, marker="s", label="F1 macro")
eixos[0].set(xlabel="Época", ylabel="Métrica", ylim=(0.88, 0.95), title="Desempenho de avaliação")
eixos[0].legend()
eixos[0].grid(alpha=0.3)

eixos[1].plot(epocas, perdas, marker="o", color="tab:red")
eixos[1].set(xlabel="Época", ylabel="Loss", title="Loss de avaliação")
eixos[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 7.1 Avaliação detalhada

A célula seguinte recalcula as métricas, o relatório por classe e a matriz de confusão usando o checkpoint salvo. Ela realiza somente inferência, não treinamento.


In [ ]:
if MODEL_DIR.exists():
    metricas_finais = trainer.evaluate()
    resultado_predicoes = trainer.predict(encoded_dataset["evaluation"])
    y_true = resultado_predicoes.label_ids
    y_pred = np.argmax(resultado_predicoes.predictions, axis=1)

    print(f"Acurácia: {metricas_finais['eval_accuracy']:.2%}")
    print(f"F1 macro: {metricas_finais['eval_f1_macro']:.2%}")
    print(classification_report(y_true, y_pred, target_names=labels, digits=3, zero_division=0))

    fig, ax = plt.subplots(figsize=(16, 14))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        labels=list(range(num_labels)),
        display_labels=labels,
        normalize="true",
        include_values=False,
        xticks_rotation=90,
        cmap="Blues",
        ax=ax,
        colorbar=False,
    )
    ax.set_title("Matriz de confusão normalizada — ESC-50")
    plt.tight_layout()
    plt.show()
else:
    print("Checkpoint local não disponível; permanecem válidos os resultados registrados acima.")


### 7.2 Interpretação dos erros

A maior parte das classes apresentou desempenho elevado. Entre as categorias mais desafiadoras no fold avaliado estão `helicopter`, `airplane`, `pig`, `washing_machine` e `door_wood_creaks`. Essas confusões são plausíveis porque algumas classes compartilham padrões contínuos de baixa frequência ou possuem poucos eventos acústicos distintivos dentro do clipe.


## 8. Inferência

A função abaixo recebe um áudio, aplica reamostragem quando necessário e retorna as cinco classes mais prováveis.


In [ ]:
def predict_audio(audio_array, sampling_rate, top_k=5):
    if not MODEL_DIR.exists():
        raise FileNotFoundError("O checkpoint treinado não está disponível.")

    audio_array = np.asarray(audio_array, dtype=np.float32)
    if sampling_rate != feature_extractor.sampling_rate:
        audio_array = librosa.resample(
            audio_array,
            orig_sr=sampling_rate,
            target_sr=feature_extractor.sampling_rate,
        )
        sampling_rate = feature_extractor.sampling_rate

    inputs = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt",
    )
    inputs = {nome: tensor.to(DEVICE) for nome, tensor in inputs.items()}
    model.to(DEVICE).eval()
    with torch.inference_mode():
        probabilidades = torch.softmax(model(**inputs).logits, dim=-1)[0]

    valores, indices = torch.topk(probabilidades, k=top_k)
    return [
        {
            "classe": model.config.id2label[indice.item()],
            "probabilidade": valor.item(),
        }
        for valor, indice in zip(valores.cpu(), indices.cpu())
    ]


### 8.1 Exemplo do conjunto de avaliação


In [ ]:
if MODEL_DIR.exists():
    exemplo = dataset["evaluation"][0]
    previsoes = predict_audio(
        exemplo["audio"]["array"],
        exemplo["audio"]["sampling_rate"],
    )
    print("Classe correta:", exemplo["category"])
    print("Top 5 previsões:")
    for previsao in previsoes:
        print(f"  {previsao['classe']:<25} {previsao['probabilidade']:.2%}")
else:
    print("Exemplo de inferência indisponível sem o checkpoint local.")


### 8.2 Visualização Log-Mel


In [ ]:
if MODEL_DIR.exists():
    audio_array = np.asarray(exemplo["audio"]["array"], dtype=np.float32)
    sampling_rate = exemplo["audio"]["sampling_rate"]
    espectrograma_mel = librosa.feature.melspectrogram(
        y=audio_array, sr=sampling_rate, n_fft=1024, hop_length=160, n_mels=128
    )
    espectrograma_db = librosa.power_to_db(espectrograma_mel, ref=np.max)

    plt.figure(figsize=(12, 5))
    librosa.display.specshow(
        espectrograma_db,
        sr=sampling_rate,
        hop_length=160,
        x_axis="time",
        y_axis="mel",
        cmap="magma",
    )
    plt.colorbar(format="%+2.0f dB")
    plt.title(f"Espectrograma Log-Mel — classe: {exemplo['category']}")
    plt.tight_layout()
    plt.show()


### 8.3 Áudio externo

Para testar um arquivo próprio, coloque-o em `data/audio_exemplo.wav`. O arquivo não faz parte do repositório.


In [ ]:
ARQUIVO_AUDIO = PROJECT_ROOT / "data" / "audio_exemplo.wav"

if ARQUIVO_AUDIO.exists() and MODEL_DIR.exists():
    audio_proprio, sr_proprio = librosa.load(
        ARQUIVO_AUDIO,
        sr=feature_extractor.sampling_rate,
        mono=True,
    )
    for previsao in predict_audio(audio_proprio, sr_proprio):
        print(f"{previsao['classe']:<25} {previsao['probabilidade']:.2%}")
else:
    print("Adicione data/audio_exemplo.wav para realizar este teste opcional.")


## 9. Limitações

- Foi avaliado apenas um dos cinco folds oficiais. O protocolo completo requer cinco treinamentos, alternando o fold de avaliação.
- O fold 5 foi acompanhado ao final de cada época e também determinou o melhor checkpoint; portanto, os resultados devem ser interpretados como avaliação experimental, não como teste cego independente.
- O modelo foi pré-treinado no AudioSet, de modo que o desempenho não representa aprendizado exclusivamente a partir do ESC-50.
- A inferência é fechada nas 50 classes: áudios externos a esse conjunto ainda receberão uma dessas categorias.
- O dataset é pequeno e não representa toda a diversidade acústica de ambientes reais.


## 10. Conclusões

O uso de *transfer learning* com o Audio Spectrogram Transformer mostrou-se eficaz para classificação de sons ambientais. A reamostragem para 16 kHz, a representação Log-Mel e a adaptação da cabeça de classificação permitiram atingir **92,75% de acurácia** e **92,64% de F1 macro** no fold avaliado.

A análise por classe indica desempenho elevado na maior parte das categorias, embora sons com características espectrais semelhantes ainda apresentem confusões. Como continuidade, recomenda-se executar a validação cruzada completa nos cinco folds e avaliar técnicas de aumento de dados.


## Referências

1. PICZAK, K. J. **ESC: Dataset for Environmental Sound Classification**. ACM Multimedia, 2015. [DOI](https://doi.org/10.1145/2733373.2806390).
2. GONG, Y.; CHUNG, Y.-A.; GLASS, J. **AST: Audio Spectrogram Transformer**. Interspeech, 2021. [Artigo](https://arxiv.org/abs/2104.01778).
3. MIT. **AST fine-tuned on AudioSet**. [Hugging Face](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593).
4. Documentação das bibliotecas [PyTorch](https://pytorch.org/), [Transformers](https://huggingface.co/docs/transformers/) e [Librosa](https://librosa.org/).
